# Scraping a Real Estate Dataset for Islamabad

The agent we are going to built needs to answer inquiries about real estate properties in Islamanad but there's no ready-made "Islamabad real estate CSV" sitting around on Kaggle, so this notebook walks through building one: picking a source that's actually permitted to scrape, understanding its data shape, and turning it into `data/islamabad_properties.csv`.

The production version of everything here lives in [`scripts/graana_scraper.py`](../scripts/graana_scraper.py) — a standalone, tested script. This notebook imports and exercises its real functions rather than reimplementing them, so what you see here is exactly what runs in production.

## Step 1 — The obvious choice doesn't work: zameen.com

zameen.com is Pakistan's biggest property portal, so it's the natural first target. Let's check what its `robots.txt` actually allows before writing a single line of scraping code — skipping that check is how you end up building something you can't legally run.

In [1]:
import urllib.robotparser as robotparser

rp = robotparser.RobotFileParser()
rp.set_url("https://www.zameen.com/robots.txt")
rp.read()

test_url = "https://www.zameen.com/Houses_Property/Islamabad_Capital-1562-1.html"
print("Allowed?", rp.can_fetch("*", test_url))

Allowed? False


`False`. And it's not a narrow, incidental block — zameen.com's `robots.txt` disallows `/Islamabad*`, plus the equivalent path for essentially every major Pakistani city (`/Karachi*`, `/Lahore*`, `/Rawalpindi*`, ...). That's a deliberate, blanket policy against crawling their city-level listing pages, not an accident. Respecting that is a hard boundary here — this notebook doesn't try to route around it.

## Step 2 — Finding a source that actually allows it

A few other Pakistani property portals exist. Checking their `robots.txt` the same way, rather than assuming any of them are fair game:

In [2]:
candidates = ["graana.com", "pakistanproperty.com", "aarz.pk"]

for site in candidates:
    rp = robotparser.RobotFileParser()
    rp.set_url(f"https://www.{site}/robots.txt")
    try:
        rp.read()
        allowed = rp.can_fetch("*", f"https://www.{site}/")
    except Exception:
        allowed = True  # no readable robots.txt -> nothing explicitly disallowed
    print(f"{site:24s} crawling allowed: {allowed}")

graana.com               crawling allowed: True


pakistanproperty.com     crawling allowed: False


aarz.pk                  crawling allowed: True


Two workable options turn up:

- **graana.com** — `robots.txt` has `User-agent: *` / `Allow: /`, with no per-city block anywhere in the file (the only `Disallow: /` rules target specific AI-training crawlers like `Amazonbot`/`Bytespider`, not general crawling).
- **pakistanproperty.com** — similarly permissive, just blocking admin/query-string paths.
- **aarz.pk** — no `robots.txt` at all (technically unrestricted, though that's a weaker signal than an explicit `Allow`).

graana.com has real, substantial Islamabad listings and an explicit `Allow: /`, so that's the target.

## Step 3 — graana.com is server-rendered: the `__NEXT_DATA__` trick

The instinct here might be "it's a modern web app, I'll need a headless browser (Playwright) to render the JavaScript." That instinct was tested and turned out to be wrong for this site: graana.com is built with Next.js in **server-rendered** mode, meaning the HTML it sends back on the very first request already contains the full listing data as a JSON blob — no browser, no waiting for client-side rendering.

That blob lives in a `<script id="__NEXT_DATA__">` tag. Let's fetch one search page and pull it out.

In [3]:
import sys
sys.path.append("../scripts")

import requests
from graana_scraper import extract_next_data

SEARCH_URL = "https://www.graana.com/sale/house-sale-islamabad-1/"

r = requests.get(SEARCH_URL, headers={"User-Agent": "Mozilla/5.0"}, timeout=15)
print("HTTP", r.status_code, "-", len(r.text), "bytes of HTML")

data = extract_next_data(r.text)
print("Top-level keys:", list(data.keys()))
print("props.pageProps keys:", list(data["props"]["pageProps"].keys()))

HTTP 200 - 609844 bytes of HTML
Top-level keys: ['props', 'page', 'query', 'buildId', 'isFallback', 'dynamicIds', 'gssp', 'scriptLoader']
props.pageProps keys: ['isMobileAgent', 'properties', 'propertiesCount', 'seoData', 'initialState']


`props.pageProps` is where the interesting stuff lives: a `properties` array (the actual listings on this page), a `propertiesCount` (total listings matching this search, across all pages), and a nested `initialState.filter.filter` object holding the current query's `page`/`pageSize`.

Let's look at one full listing record.

In [4]:
page_props = data["props"]["pageProps"]
properties = page_props["properties"]

print(f"{len(properties)} listings on this page, {page_props['propertiesCount']} total\n")
import json as _json
print(_json.dumps(properties[0], indent=2, ensure_ascii=False))

30 listings on this page, 537 total

{
  "id": 1552161,
  "purpose": "buy",
  "type": "residential",
  "customTitleGenerated": false,
  "subtype": "house",
  "areaId": 1872,
  "cityId": 1,
  "price": "21000000",
  "size": 5,
  "sizeUnit": "marla",
  "userId": "e4e89458-c0d1-703d-64fd-bab16f3f0a0f",
  "status": "published",
  "bed": 4,
  "bath": 4,
  "agencyId": null,
  "customTitle": "5 Marla house for sale",
  "propsureId": null,
  "createdAt": "2026-08-18T18:32:13.573Z",
  "totalImages": "7",
  "name": "Abu Zar",
  "area": {
    "id": 1872,
    "name": "Faisal Hills"
  },
  "city": {
    "id": 1,
    "name": "Islamabad"
  },
  "propertyImages": [
    {
      "id": 11010152,
      "url": "/images/original/efefb873180ed1a767b9581fb8ec3658",
      "type": "cover",
      "updatedAt": "2026-08-18T18:33:20.155Z"
    },
    {
      "id": 11010153,
      "url": "/images/original/247b25b1bd4e1acbc1927f882cb40428",
      "type": "image",
      "updatedAt": "2026-08-18T18:33:20.152Z"
    },
   

Clean, structured data straight from the source: `price` (a numeric string, in PKR), `size` + `sizeUnit` (e.g. `22` / `"marla"`), `bed`, `bath`, and a nested `area.name` / `city.name` for location — no HTML-card parsing, no CSS selectors that'll break on the next redesign.

## Step 4 — Pagination

541 listings ÷ 30 per page is 18.033~19 pages. The question is how to actually ask for page 2. Clicking through the site's own "Next" button (via a real browser, just for this investigation) revealed the URL changes to include `?pageSize=30&page=2` — and, importantly, that same query string works on a **plain HTTP request**, no JavaScript required.

`graana_scraper.py`'s `with_query()` helper builds that URL, and `extract_page()` wraps the whole "fetch + pull out `__NEXT_DATA__` + shape it into rows" flow, auto-detecting how many pages exist from `propertiesCount`.

In [5]:
from graana_scraper import with_query, extract_page

page_2_url = with_query(SEARCH_URL, pageSize=30, page=2)
print("Page 2 URL:", page_2_url)

r2 = requests.get(page_2_url, headers={"User-Agent": "Mozilla/5.0"}, timeout=15)
rows, count, page_size = extract_page(r2.text, page_2_url)

print(f"\nparsed {len(rows)} rows, propertiesCount={count}, pageSize={page_size}")
print("first listing id on page 2:", rows[0]["id"], "-", rows[0]["title"])

Page 2 URL: https://www.graana.com/sale/house-sale-islamabad-1/?pageSize=30&page=2



parsed 30 rows, propertiesCount=537, pageSize=30
first listing id on page 2: 1547142 - 10 Marla House for Sale


A different first listing than page 1, same `propertiesCount` — confirming this is genuinely page 2 of the same 547-listing query, not a different search.

## Step 5 — Being a polite scraper

Finding permission in `robots.txt` doesn't mean hammering the site. `graana_scraper.py`'s `Fetcher` class bakes in the basics:

- Loads and checks `robots.txt` before every request (fails closed if it can't be read).
- Rate-limits with a delay + jitter between requests.
- Retries with exponential backoff on `429`/`503`, and on plain connection errors.
- Identifies itself with a descriptive `User-Agent` rather than pretending to be a browser.

In [6]:
from graana_scraper import Fetcher

fetcher = Fetcher(delay=2.0)
print("robots.txt allows our target URL:", fetcher.allowed(SEARCH_URL))

robots.txt allows our target URL: True


## Step 6 — Unit conversion: marla/kanal → square metres

Pakistani real estate is priced and sized in local units — *marla* and *kanal* — not square metres. This dataset uses a square-metres scale (column `sqm`), so sizes get converted to match (1 Marla = 25 sq. yards = 20.90 m²; 1 Kanal = 20 Marla), verified against real listings on the site.

In [7]:
from graana_scraper import size_to_sqm

examples = [(22, "marla"), (1, "kanal"), (1200, "sqft"), (5, "marla")]
for size, unit in examples:
    print(f"{size} {unit:6s} -> {size_to_sqm(size, unit):.2f} m²")

22 marla  -> 459.87 m²
1 kanal  -> 418.06 m²
1200 sqft   -> 111.48 m²
5 marla  -> 104.52 m²


## Step 7 — Price: kept in PKR, and a sanity check on what `price` actually means

Two decisions worth making explicit:

1. **No currency conversion.** The agent is simply told the currency is PKR, so `price` is graana's raw PKR value, parsed straight from the JSON — no conversion needed.
2. **Is `price` ever a *rental* rate in disguise?** Only `purpose: "buy"` listings are scraped (never `"rent"`), so `price` should always be a one-time sale price. Worth actually checking that assumption against the real data rather than just trusting it.

In [8]:
from graana_scraper import parse_price_pkr

# Every listing on this page should be a sale, never a rental.
purposes = {p.get("purpose") for p in properties}
print("purposes present on this page:", purposes)

print("parse_price_pkr('85000000') ->", parse_price_pkr("85000000"))
print("parse_price_pkr(None)       ->", parse_price_pkr(None))

purposes present on this page: {'buy'}
parse_price_pkr('85000000') -> 85000000
parse_price_pkr(None)       -> None


`purposes` comes back as `{'buy'}` — confirmed, this search page never mixes in rentals.

There's a subtler trap, though: some listings' free-text *descriptions* mention rental figures as a side note (an owner advertising a property's rental-income potential), like:

> *"Rental Value: Per day: 12000 - 15000, Per Month: 1.25 - 1.50 lacs"*

That text lives inside `description`, completely separate from the structured `price` field — and the scale gives it away instantly: a mentioned daily/monthly rental figure (₨12,000–150,000) is two orders of magnitude smaller than an actual Islamabad house sale price (tens of millions of PKR). No risk of the two getting confused in the `price` column itself.

## Step 8 — Descriptions: real, enriched, or synthesized

`description` is the one free-text field in this dataset, and it deserves more care than the other columns. This data will later get indexed for *semantic* search — comparing listings by what their text actually says, not just filtering on structured fields like price or bed count. graana's `customTitle` field is short (`"22 Marla house for sale"`), and dozens of listings share almost that exact same title, so relying on it alone would make most listings look nearly indistinguishable to that kind of search.

Three strategies, tried in order, give every listing something more substantive to work with:

- **`build_description()`** — a fallback, synthesized from structured fields on the *search* page (`customTitle` + bed/bath + area/city). Cheap: no extra request, but generic — really just the structured fields restated as a sentence.
- **`extract_description()`** — the *real* description text an agent wrote on the listing's own detail page. The best case, but many listings simply don't have one (left blank).
- **`describe_extras()`** — when there's no real description, the detail page often still has structured extras that never made it into the free text: `condition` and feature flags (parking, room counts, utilities, nearby amenities). These get turned into readable sentence fragments and appended to the synthesized fallback, so even a "generic" listing ends up with real, differentiating detail — at no extra request cost, since the detail page is already being fetched for the real-description attempt.

In [9]:
from graana_scraper import build_description, extract_description, extract_detail_extras, describe_extras

sample_prop = properties[0]
print("Synthesized:", build_description(sample_prop))

detail_url = f"https://www.graana.com/property/listing-{sample_prop['id']}/"
r3 = requests.get(detail_url, headers={"User-Agent": "Mozilla/5.0"}, timeout=15, allow_redirects=True)
real_desc = extract_description(r3.text)
print("\nReal (graana):", real_desc or "<empty - agent left no description>")

extras_text = describe_extras(extract_detail_extras(r3.text))
print("\nExtras (condition/features/nearby):", extras_text or "<none for this listing>")

Synthesized: 5 Marla house for sale, 4 bedrooms, 4 bathrooms, located in Faisal Hills, Islamabad



Real (graana): HOUSE FOR SALE Faisal Hills Luxury Living in the Heart of Faisal Hills Islamabad Faisal Hills, Islamabad Block A, Street 123A beautifully designed 5 Marla House offering comfort, elegance, and exceptional value. Located on a prime location of Block A Faisal Hills 5300 series, this home is perfect for families seeking a premium lifestyle. 4 Master Bedrooms with Attached Bathrooms Double Unit House 2 Stylish Drawing Rooms 2 Spacious TV Lounges Huge Car Porch Own Water Bore (24/7 Water Supply) Arch and Sun face Modern Architecture Prime Location Excellent Investment OpportunityYour dream home is waiting for you. Contact now to schedule a visit and experience luxury living.0301-5258451

Extras (condition/features/nearby): Condition: Brand new.


All three versions get produced for every listing where possible; `description_source` in the final dataset records which one actually won out (`"graana"` for a real description, `"enriched"` when extras got appended to the fallback, `"synthesized"` when neither was available), so it's auditable per-row rather than a silent guess.

## Step 9 — Running it for real

Everything above is `graana_scraper.py`'s internals, exercised piece by piece. Putting it together with `scrape()` runs the full pipeline: paginate, parse, convert units, decide on a description, and yield one `Listing` per row. Here it's run for a single page as a fast, live demo — the full dataset (all 547 listings, `--details` enabled) already lives at `data/islamabad_properties.csv` and took about 20 minutes to build, since fetching each real description is one extra polite, rate-limited request.

In [10]:
from graana_scraper import scrape
import pandas as pd

demo_fetcher = Fetcher(delay=1.0)  # short delay since this is just 1 page for the demo
demo_rows = list(scrape(SEARCH_URL, pages=1, fetcher=demo_fetcher, want_details=False))

demo_df = pd.DataFrame([r.__dict__ for r in demo_rows])
demo_df[["id", "price", "baths", "rooms", "sqm", "location", "description_source"]].head()

,id,price,baths,rooms,sqm,location,description_source
0,1552161,21000000,4,4,105,Faisal Hills,synthesized
1,1551889,85000000,6,5,460,Emaar Canyon Views,synthesized
2,1551528,23000000,7,5,105,Ali Pur,synthesized
3,1551078,59500000,5,5,209,Bahria Enclave,synthesized
4,1551075,29500000,4,4,105,Bahria Enclave,synthesized


## Step 10 — The dataset the agent actually uses

`data/islamabad_properties.csv` is the full run: `id, price, baths, rooms, sqm, description, location`, plus one audit column, `description_source`.

A few other candidate columns were tried along the way and dropped again:

- **Geo coordinates (`lat`/`lng`)** and the **raw per-category feature flags** — it wasn't clear how the voice agent would actually use raw coordinates, and an empty feature dict doesn't reliably mean "this property lacks parking/a lawn/etc." — it just means the listing agent didn't mention it. Treating that absence as a queryable "no" would be misleading. The feature data still earns its keep, though: it's folded into `description` itself via `describe_extras()` wherever a real description was missing (that's the `"enriched"` case in `description_source`), which is a much better fit for free text than for a structured column.
- **`address`** — turned out to be redundant with `location`: identical on 96% of listings, and blank (not more specific) on the rest.

In [11]:
islamabad_df = pd.read_csv("../data/islamabad_properties.csv")

print(f"{len(islamabad_df)} listings\n")
print("description_source breakdown:")
print(islamabad_df["description_source"].value_counts())
print(f"\nprice range (PKR): {islamabad_df['price'].min():,} - {islamabad_df['price'].max():,}")
print(f"size range (m²):   {islamabad_df['sqm'].min()} - {islamabad_df['sqm'].max()}")

islamabad_df.head()

541 listings

description_source breakdown:
description_source
enriched       353
graana         125
synthesized     63
Name: count, dtype: int64

price range (PKR): 4,200,000 - 750,000,000
size range (m²):   21 - 941


,id,price,baths,rooms,sqm,description,location,size_raw,url,title,description_source,scraped_at
0,1551889,85000000,6,5,460,The property is located in a premium and secur...,Emaar Canyon Views,22 marla,https://www.graana.com/property/listing-1551889/,22 Marla house for sale,graana,2026-08-18T02:48:21+0500
1,1551528,23000000,7,5,105,🏡 5 Marla Beautiful Double & Half Story House ...,Ali Pur,5 marla,https://www.graana.com/property/listing-1551528/,5 Marla House for Sale,graana,2026-08-18T02:48:24+0500
2,1551078,59500000,5,5,209,"10 Marla House for Sale, 5 bedrooms, 5 bathroo...",Bahria Enclave,10 marla,https://www.graana.com/property/listing-1551078/,10 Marla House for Sale,enriched,2026-08-18T02:48:29+0500
3,1551075,29500000,4,4,105,"5 Marla House for Sale, 4 bedrooms, 4 bathroom...",Bahria Enclave,5 marla,https://www.graana.com/property/listing-1551075/,5 Marla House for Sale,enriched,2026-08-18T02:48:33+0500
4,1551068,28500000,4,4,105,"5 Marla House for Sale, 4 bedrooms, 4 bathroom...",Bahria Enclave,5 marla,https://www.graana.com/property/listing-1551068/,5 Marla House for Sale,enriched,2026-08-18T02:48:37+0500


## Next steps

This dataset feeds into the Superlinked pipeline in `3_superlinked_property_search.ipynb`. One thing that notebook will need updated when it switches over to this dataset: it currently expects a column named `sqft` (inherited from an earlier Kaggle dataset that used that name despite holding square metres) — this dataset names the same square-metres data `sqm` instead, since that earlier dataset is going away.